# Test Mtb readout integration

In [9]:
import napari
import zarr
import pandas as pd
import numpy as np
from pathlib import Path

# --- Paths ---
zarr_path = Path("/mnt/DATA3/BPP0050/BPP0050-1-Live-cell-to4i_Live-1__2025-04-09T18_25_04-Measurement 1/live_1.zarr")  # adjust this
pos = "3_3"
images_path = zarr_path / pos / "images" / "0"
labels_path = zarr_path / pos / "labels" / "trackastra_labels" / "0"
tracks_path = zarr_path / pos / "labels" / "trackastra_labels" / "tables" / "track"
zarr_root = zarr.open_group(str(zarr_path / pos), mode="r")

# --- Load data ---
image_array = zarr_root["images"]["0"]    
seg_array = zarr_root["labels"]["trackastra_labels"]["0"]

# Load first 3 frames
image_preview = image_array.images[:48].max(axis=2)     # (3, C, Y, X)
seg_preview = seg_array[:48]         # (3, Y, X)

# Load and filter track table
track_columns = [f.name for f in tracks_path.iterdir() if f.is_dir()]
track_data = {
    col: zarr.open(tracks_path / col, mode='r')[:]
    for col in track_columns
}
track_df = pd.DataFrame(track_data)

track_subset = track_df[track_df["t"] < 20]  # filter first 3 frames
coords = track_subset[["track_id", "t", "y", "x"]].values
props = {col: track_subset[col].values for col in track_subset.columns if col not in ["t", "y", "x"]}

# --- Launch Napari ---
viewer = napari.Viewer()
viewer.add_image(image_preview, channel_axis=1,)  # adjust channel index if needed
viewer.add_labels(seg_preview, name="Segmentation")
viewer.add_tracks(coords, name="Tracks",)# size=6, face_color='track_id', ndim=3)
napari.run()

In [8]:
image_array['images']

<zarr.core.Array '/images/0/images' (48, 3, 3, 6911, 6911) uint16 read-only>

In [5]:
seg_preview.shape, image_preview.shape

((20, 6911, 6911), (20, 3, 3, 6911, 6911))

In [ ]:
track_df = pd.DataFrame(track_data)

track_subset = track_df[track_df["t"] < 20]  # filter first 3 frames
coords = track_subset[["track_id", "t", "y", "x"]].values
props = {col: track_subset[col].values for col in track_subset.columns if col not in ["t", "y", "x"]}

# --- Launch Napari ---
viewer = napari.Viewer()
viewer.add_image(image_preview, channel_axis=1,)  # adjust channel index if needed
viewer.add_labels(seg_preview, name="Segmentation")
viewer.add_tracks(coords, name="Tracks",)# size=6, face_color='track_id', ndim=3)
napari.run()